# Week 5 — Deep Learning Model Development and Evaluation
**Student:** Vikhyath Bharadwaj K S  
**Project:** Handwritten digit classification with a convolutional neural network  

This notebook trains a compact PyTorch CNN on scikit-learn’s bundled copy of the UCI Optical Recognition of Handwritten Digits dataset. The bundled source makes the analysis reproducible without a network download. Fashion-MNIST was considered but could not be fetched in the restricted runtime.

## 1. Problem and dataset
Classify each 8×8 grayscale image (pixel intensities 0–16) as one of ten handwritten digits. This is supervised, single-label, multiclass image classification. The dataset contains 1,797 examples (179 per digit, with a few class-count rounding differences). Images are normalized to [0,1]. We create stratified train, validation, and test subsets (70/15/15; seed 42).

In [ ]:
from sklearn.datasets import load_digits
import numpy as np
data=load_digits()
print('Images:', data.images.shape, 'Labels:', data.target.shape)
print('Pixel range:', data.images.min(), 'to', data.images.max())
print('Class counts:', np.bincount(data.target))

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(2,5,figsize=(9,4))
for digit,a in enumerate(ax.flat):
    i=np.flatnonzero(data.target==digit)[0]; a.imshow(data.images[i],cmap='gray'); a.set_title(f'Label {digit}'); a.axis('off')
plt.tight_layout()

## 2. Architecture and training setup
Two convolution blocks use 3×3 convolutions (16 then 32 filters), ReLU, and 2×2 max pooling. The 32×2×2 feature map feeds a 64-unit ReLU layer, 20% dropout, and ten logits. Cross-entropy loss and Adam (learning rate 0.001) are used with batch size 64 for 30 epochs. Validation loss selects the saved best checkpoint. No augmentation is used because the dataset is small and its orientation/shape transformations may alter the digit label.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'src'/'week5'/'train.py').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.week5.train import DigitsCNN, main
model=DigitsCNN()
print(model)

## 3. Train and evaluate
The cell below performs the stratified split, trains the CNN, selects the checkpoint by validation loss, evaluates once on the held-out test partition, and writes metrics, weights, and visualizations into the repository. The test data are not used for model selection.

In [ ]:
metrics=main(epochs=30,batch_size=64)
print(f"Test accuracy: {metrics['test_accuracy']:.4f}")
print(f"Test macro F1: {metrics['test_macro_f1']:.4f}")
print(f"Test weighted F1: {metrics['test_weighted_f1']:.4f}")

In [ ]:
from IPython.display import display, Image
for name in ['training_curves.png','confusion_matrix.png','architecture.png','misclassified_examples.png']:
    print(name)
    display(Image(filename=str(ROOT/'visualizations'/'week5'/name)))

## 4. Learning curves and error analysis

In [ ]:
import json
run=json.loads((ROOT/'models'/'week5'/'metrics.json').read_text(encoding='utf-8'))
for label in run['labels']:
    row=run['classification_report'][label]
    print(f"{label}: precision={row['precision']:.3f}, recall={row['recall']:.3f}, F1={row['f1-score']:.3f}, n={int(row['support'])}")

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(
    np.array(data.target)[np.array_split(np.arange(len(data.target)), 1)[0]][:0] if False else [], []))

## 5. Interpretation and limitations
Digits is a compact benchmark (1,797 observations and 8×8 resolution), not a deployment dataset. Results can vary by framework/hardware despite fixed seeds, and a single split has sampling uncertainty. Handwritten style and the low resolution constrain generalization. Compare per-class precision/recall and inspect the saved confusion matrix before drawing conclusions. Model and run artifacts are intentionally excluded from Git; regenerate them with `python src/week5/train.py`.